# Step 17 — Pairwise E5 cross-encoder reranker

Goal: train a lighter-than-Qwen reranker that learns ordinal relevance by comparing papers, not only by regressing each label independently.

This notebook uses `intfloat/e5-large-v2` as a cross-encoder-style scorer: a fixed task query and each paper are packed into one sequence, then a scalar relevance score is produced. Training uses pairwise ranking loss plus a small pointwise calibration loss so the final scores can still be thresholded into labels 1-5.

Outputs match the existing anchor schema: `oof_scores.csv`, `public_scores.csv`, `private_scores.csv`, `metrics.json`, and a submission CSV. Run on Colab A100/H100/H200.

## 1. GPU + dependencies

In [ ]:
!nvidia-smi
!pip install -q --upgrade "transformers>=4.41" "accelerate>=0.30" sentencepiece scikit-learn pandas "numpy<2" scipy

## 2. Data setup

In [ ]:
import os, pathlib, zipfile, json, time, gc, random
import numpy as np
import pandas as pd

def detect_platform():
    if 'COLAB_RELEASE_TAG' in os.environ or 'COLAB_GPU' in os.environ:
        return 'colab'
    if 'KAGGLE_KERNEL_RUN_TYPE' in os.environ or pathlib.Path('/kaggle/working').exists():
        return 'kaggle'
    return 'local'

PLATFORM = detect_platform()
WORK = pathlib.Path('/content/work') if PLATFORM == 'colab' else pathlib.Path('/kaggle/working/asp_work') if PLATFORM == 'kaggle' else pathlib.Path.cwd() / 'work'
DATA = WORK / 'data'
OUT = WORK / 'outputs'
RUN_DIR = OUT / 'pairwise_e5_reranker'
for d in [WORK, DATA, OUT, RUN_DIR]:
    d.mkdir(parents=True, exist_ok=True)

def find_zip():
    env_zip = os.environ.get('ASP_DATA_ZIP')
    if env_zip and pathlib.Path(env_zip).exists():
        return pathlib.Path(env_zip)
    for base in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        for pat in ('asp_data*.zip', 'asp_data.zip'):
            hits = sorted(base.glob(pat))
            if hits:
                return hits[0]
        if (base / '.git').exists():
            break
    if pathlib.Path('/kaggle/input').exists():
        hits = sorted(pathlib.Path('/kaggle/input').rglob('asp_data*.zip')) or sorted(pathlib.Path('/kaggle/input').rglob('*.zip'))
        if hits:
            return hits[0]
    return None

zip_path = find_zip()
if zip_path is None and PLATFORM == 'colab':
    from google.colab import files
    uploaded = files.upload()
    for name, content in uploaded.items():
        target = WORK / name
        target.write_bytes(content)
        if name.lower().endswith('.zip'):
            zip_path = target
            break
if zip_path is None:
    raise FileNotFoundError('No asp_data zip found. Upload it or set ASP_DATA_ZIP.')
print('using zip:', zip_path)
with zipfile.ZipFile(zip_path) as zf:
    zf.extractall(DATA)
print('extracted to', DATA)

## 3. Load data + abstracts

In [ ]:
from pathlib import Path

train = pd.read_csv(DATA / 'train.csv')
public = pd.read_csv(DATA / 'public_test.csv')
private = pd.read_csv(DATA / 'private_test.csv')
sample = pd.read_csv(DATA / 'Test_Submission.csv')
abstracts_file = DATA / 'abstracts_merged_v2.csv'
if not abstracts_file.exists():
    abstracts_file = DATA / 'abstracts_merged_v3.csv'
abstracts = pd.read_csv(abstracts_file)
print('abstract cache:', abstracts_file.name)

abs_map = abstracts[['source_split', 'id', 'abstract', 'has_abstract']]
def attach(df, split):
    df = df.copy()
    df['source_split'] = split
    out = df.merge(abs_map, on=['source_split', 'id'], how='left')
    out['abstract'] = out['abstract'].fillna('')
    out['has_abstract'] = out['has_abstract'].fillna(False).astype(bool)
    return out.reset_index(drop=True)

train_full = attach(train, 'train')
public_full = attach(public, 'public_test')
private_full = attach(private, 'private_test')
for name, df in [('train', train_full), ('public', public_full), ('private', private_full)]:
    print(name, len(df), 'abstract coverage', round(df['has_abstract'].mean(), 3))

## 4. Tokenizer, text format, datasets

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel

MODEL_NAME = 'intfloat/e5-large-v2'
MAX_LEN = 384
BATCH_PAIR = 6
BATCH_EVAL = 24
GRAD_ACCUM_STEPS = 2
PAIRS_PER_EPOCH = 6000
PAIR_MIN_GAP = 1
RANK_LOSS_WEIGHT = 1.0
POINTWISE_LOSS_WEIGHT = 0.30
TASK_QUERY = 'query: Rank how relevant this research paper is to answer set programming, logic programming, symbolic AI, knowledge representation, constraints, planning, reasoning, or related ASP applications.'

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
SEP = tokenizer.sep_token or '[SEP]'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
AMP_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16
print('device =', device, 'amp =', 'bf16' if USE_BF16 else 'fp16', 'sep =', SEP)

def paper_text(row):
    title = '' if pd.isna(row.get('title', '')) else str(row.get('title', '')).strip()
    abstract = '' if pd.isna(row.get('abstract', '')) else str(row.get('abstract', '')).strip()
    authors = '' if pd.isna(row.get('authors', '')) else str(row.get('authors', '')).strip()
    body = f'title: {title}'
    if abstract:
        body += f' {SEP} abstract: {abstract}'
    if authors:
        body += f' {SEP} authors: {authors}'
    return body

def make_cross_text(row):
    return TASK_QUERY + ' ' + SEP + ' passage: ' + paper_text(row)

class PairDataset(Dataset):
    def __init__(self, df, pairs_per_epoch, seed):
        self.df = df.reset_index(drop=True)
        self.texts = [make_cross_text(r) for _, r in self.df.iterrows()]
        self.labels = self.df['Label'].astype(np.float32).to_numpy()
        self.by_label = {lab: np.where(self.labels.astype(int) == lab)[0] for lab in range(1, 6)}
        self.pairs_per_epoch = pairs_per_epoch
        self.rng = np.random.default_rng(seed)
        self.valid_hi = [lab for lab in range(2, 6) if len(self.by_label[lab]) > 0]

    def __len__(self):
        return self.pairs_per_epoch

    def __getitem__(self, idx):
        for _ in range(20):
            hi_lab = int(self.rng.choice(self.valid_hi))
            lo_choices = [lab for lab in range(1, hi_lab) if hi_lab - lab >= PAIR_MIN_GAP and len(self.by_label[lab]) > 0]
            if lo_choices:
                lo_lab = int(self.rng.choice(lo_choices))
                hi_idx = int(self.rng.choice(self.by_label[hi_lab]))
                lo_idx = int(self.rng.choice(self.by_label[lo_lab]))
                return {'hi_text': self.texts[hi_idx], 'lo_text': self.texts[lo_idx], 'hi_label': self.labels[hi_idx], 'lo_label': self.labels[lo_idx]}
        raise RuntimeError('Could not sample a valid pair.')

class PaperDataset(Dataset):
    def __init__(self, df, with_label):
        self.df = df.reset_index(drop=True)
        self.texts = [make_cross_text(r) for _, r in self.df.iterrows()]
        self.labels = self.df['Label'].astype(np.float32).to_numpy() if with_label else None

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        item = {'text': self.texts[idx], 'idx': idx}
        if self.labels is not None:
            item['label'] = self.labels[idx]
        return item

def collate_pairs(batch):
    hi = tokenizer([b['hi_text'] for b in batch], padding=True, truncation=True, max_length=MAX_LEN, return_tensors='pt', return_token_type_ids=False)
    lo = tokenizer([b['lo_text'] for b in batch], padding=True, truncation=True, max_length=MAX_LEN, return_tensors='pt', return_token_type_ids=False)
    return {
        'hi_input_ids': hi['input_ids'], 'hi_attention_mask': hi['attention_mask'],
        'lo_input_ids': lo['input_ids'], 'lo_attention_mask': lo['attention_mask'],
        'hi_label': torch.tensor([b['hi_label'] for b in batch], dtype=torch.float32),
        'lo_label': torch.tensor([b['lo_label'] for b in batch], dtype=torch.float32),
    }

def collate_eval(batch):
    enc = tokenizer([b['text'] for b in batch], padding=True, truncation=True, max_length=MAX_LEN, return_tensors='pt', return_token_type_ids=False)
    out = {'input_ids': enc['input_ids'], 'attention_mask': enc['attention_mask'], 'idx': torch.tensor([b['idx'] for b in batch], dtype=torch.long)}
    if 'label' in batch[0]:
        out['label'] = torch.tensor([b['label'] for b in batch], dtype=torch.float32)
    return out

## 5. Cross-encoder scorer

In [ ]:
class E5CrossEncoderReranker(nn.Module):
    def __init__(self, model_name=MODEL_NAME, dropout=0.10):
        super().__init__()
        try:
            self.encoder = AutoModel.from_pretrained(model_name, torch_dtype=torch.float32)
        except TypeError:
            self.encoder = AutoModel.from_pretrained(model_name, dtype=torch.float32)
        self.encoder = self.encoder.float()
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Linear(self.encoder.config.hidden_size, 1)
        nn.init.trunc_normal_(self.head.weight, std=0.02)
        nn.init.constant_(self.head.bias, 3.0)

    def forward(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        hidden = out.last_hidden_state
        mask = attention_mask.unsqueeze(-1).float()
        pooled = (hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1.0)
        return self.head(self.dropout(pooled)).squeeze(-1)

@torch.no_grad()
def predict(model, loader):
    model.eval()
    scores = np.zeros(len(loader.dataset), dtype=np.float32)
    for batch in loader:
        ids = batch['input_ids'].to(device, non_blocking=True)
        mask = batch['attention_mask'].to(device, non_blocking=True)
        with torch.amp.autocast('cuda', dtype=AMP_DTYPE):
            pred = model(ids, mask).float().cpu().numpy()
        scores[batch['idx'].numpy()] = pred
    scores = np.where(np.isfinite(scores), scores, 3.0)
    return np.clip(scores, 1.0, 5.0)

## 6. Train one fold

In [ ]:
from sklearn.metrics import cohen_kappa_score
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup

EPOCHS = 4
LR_ENCODER = 1.0e-5
LR_HEAD = 7.0e-4
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.10
GRAD_CLIP = 1.0

def train_one_fold(train_df, valid_df, public_df, private_df, seed):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)

    pair_loader = DataLoader(PairDataset(train_df, PAIRS_PER_EPOCH, seed), batch_size=BATCH_PAIR, shuffle=False, collate_fn=collate_pairs, num_workers=2, pin_memory=True)
    valid_loader = DataLoader(PaperDataset(valid_df, with_label=True), batch_size=BATCH_EVAL, shuffle=False, collate_fn=collate_eval, num_workers=2, pin_memory=True)
    public_loader = DataLoader(PaperDataset(public_df.assign(Label=0), with_label=False), batch_size=BATCH_EVAL, shuffle=False, collate_fn=collate_eval, num_workers=2, pin_memory=True)
    private_loader = DataLoader(PaperDataset(private_df.assign(Label=0), with_label=False), batch_size=BATCH_EVAL, shuffle=False, collate_fn=collate_eval, num_workers=2, pin_memory=True)

    model = E5CrossEncoderReranker().to(device)
    no_decay = ['bias', 'LayerNorm.weight']
    enc_params = list(model.encoder.named_parameters())
    param_groups = [
        {'params': [p for n, p in enc_params if not any(nd in n for nd in no_decay)], 'lr': LR_ENCODER, 'weight_decay': WEIGHT_DECAY},
        {'params': [p for n, p in enc_params if any(nd in n for nd in no_decay)], 'lr': LR_ENCODER, 'weight_decay': 0.0},
        {'params': model.head.parameters(), 'lr': LR_HEAD, 'weight_decay': WEIGHT_DECAY},
    ]
    optim = AdamW(param_groups)
    total_steps = max(1, EPOCHS * len(pair_loader) // GRAD_ACCUM_STEPS)
    scheduler = get_linear_schedule_with_warmup(optim, int(WARMUP_RATIO * total_steps), total_steps)
    scaler = None if USE_BF16 else torch.amp.GradScaler('cuda')
    point_loss = nn.SmoothL1Loss(beta=1.0)

    best_qwk = -1.0
    best_state = None
    for epoch in range(EPOCHS):
        model.train()
        running = 0.0
        t0 = time.time()
        optim.zero_grad(set_to_none=True)
        for step, batch in enumerate(pair_loader):
            hi_ids = batch['hi_input_ids'].to(device, non_blocking=True)
            hi_mask = batch['hi_attention_mask'].to(device, non_blocking=True)
            lo_ids = batch['lo_input_ids'].to(device, non_blocking=True)
            lo_mask = batch['lo_attention_mask'].to(device, non_blocking=True)
            hi_y = batch['hi_label'].to(device, non_blocking=True)
            lo_y = batch['lo_label'].to(device, non_blocking=True)
            gap_weight = ((hi_y - lo_y) / 4.0).clamp(min=0.25, max=1.0)
            with torch.amp.autocast('cuda', dtype=AMP_DTYPE):
                hi_s = model(hi_ids, hi_mask)
                lo_s = model(lo_ids, lo_mask)
                rank_loss = (F.softplus(-(hi_s - lo_s)) * gap_weight).mean()
                calib_loss = 0.5 * (point_loss(hi_s, hi_y) + point_loss(lo_s, lo_y))
                loss = (RANK_LOSS_WEIGHT * rank_loss + POINTWISE_LOSS_WEIGHT * calib_loss) / GRAD_ACCUM_STEPS
            if scaler is not None:
                scaler.scale(loss).backward()
            else:
                loss.backward()
            if (step + 1) % GRAD_ACCUM_STEPS == 0:
                if scaler is not None:
                    scaler.unscale_(optim)
                nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                if scaler is not None:
                    scaler.step(optim); scaler.update()
                else:
                    optim.step()
                scheduler.step()
                optim.zero_grad(set_to_none=True)
            running += float(loss.item()) * GRAD_ACCUM_STEPS
        val_scores = predict(model, valid_loader)
        val_round = np.clip(np.round(val_scores), 1, 5).astype(int)
        qwk = cohen_kappa_score(valid_df['Label'].astype(int).to_numpy(), val_round, weights='quadratic')
        print(f'  epoch {epoch+1}/{EPOCHS} loss={running/max(len(pair_loader),1):.4f} val_round_QWK={qwk:.4f} time={(time.time()-t0)/60:.1f}m')
        if qwk > best_qwk:
            best_qwk = qwk
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    val = predict(model, valid_loader)
    pub = predict(model, public_loader)
    pri = predict(model, private_loader)
    del model
    gc.collect()
    torch.cuda.empty_cache()
    return val, pub, pri, best_qwk

## 7. Repeated CV

In [ ]:
from sklearn.model_selection import StratifiedKFold

FOLDS = 5
SEEDS = [252]
# If fold-1 is promising and runtime is acceptable, change to [252, 253, 254].

y_class = train_full['Label'].astype(int).to_numpy()
oof_sum = np.zeros(len(train_full), dtype=np.float64)
oof_count = np.zeros(len(train_full), dtype=np.float64)
public_sum = np.zeros(len(public_full), dtype=np.float64)
private_sum = np.zeros(len(private_full), dtype=np.float64)
fold_log = []
n_models = 0

for seed in SEEDS:
    cv = StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=seed)
    for fold, (tr_idx, va_idx) in enumerate(cv.split(train_full, y_class), start=1):
        print(f'\n=== seed={seed} fold={fold}/{FOLDS} ===')
        t0 = time.time()
        val, pub, pri, best_qwk = train_one_fold(train_full.iloc[tr_idx].reset_index(drop=True), train_full.iloc[va_idx].reset_index(drop=True), public_full, private_full, seed * 1000 + fold)
        oof_sum[va_idx] += val
        oof_count[va_idx] += 1
        public_sum += pub
        private_sum += pri
        n_models += 1
        fold_log.append({'seed': seed, 'fold': fold, 'best_round_qwk': float(best_qwk), 'minutes': round((time.time() - t0) / 60, 2)})

oof_scores = oof_sum / np.clip(oof_count, 1.0, None)
public_scores = public_sum / n_models
private_scores = private_sum / n_models
print('trained models:', n_models)
print('OOF round-QWK:', cohen_kappa_score(y_class, np.clip(np.round(oof_scores), 1, 5).astype(int), weights='quadratic'))
print(pd.DataFrame(fold_log).to_string(index=False))

## 8. Constrained threshold tuning

In [ ]:
from scipy.optimize import differential_evolution
from sklearn.metrics import f1_score, mean_absolute_error

TRAIN_DIST = pd.Series(y_class).value_counts(normalize=True).reindex([1,2,3,4,5], fill_value=0).to_numpy()

def scores_to_labels(scores, thresholds):
    return np.digitize(scores, np.sort(np.asarray(thresholds, dtype=float))) + 1

def predicted_dist(labels):
    return pd.Series(labels).value_counts(normalize=True).reindex([1,2,3,4,5], fill_value=0).to_numpy()

def tune_thresholds_constrained(y_true, oof, lambd=0.5, seed=42):
    def objective(raw):
        thr = np.sort(raw)
        gap = np.min(np.diff(thr))
        gap_pen = 0.0 if gap >= 0.03 else (0.03 - gap) * 5.0
        labels = scores_to_labels(oof, thr)
        qwk = cohen_kappa_score(y_true, labels, weights='quadratic')
        dist_pen = float(np.sum(np.abs(predicted_dist(labels) - TRAIN_DIST)))
        return -qwk + gap_pen + lambd * dist_pen
    bounds = [(1.4, 2.5), (1.8, 2.9), (2.2, 3.4), (2.6, 4.2)]
    res = differential_evolution(objective, bounds, seed=seed, maxiter=120, popsize=15, polish=True, updating='immediate', workers=1)
    thr = np.sort(res.x)
    return thr, float(cohen_kappa_score(y_true, scores_to_labels(oof, thr), weights='quadratic'))

thresholds, oof_qwk = tune_thresholds_constrained(y_class, oof_scores)
oof_pred = scores_to_labels(oof_scores, thresholds)
public_pred = scores_to_labels(public_scores, thresholds)
private_pred = scores_to_labels(private_scores, thresholds)
combined_pred = np.concatenate([public_pred, private_pred])
test_l1 = float(np.sum(np.abs(predicted_dist(combined_pred) - TRAIN_DIST)))
print('Constrained OOF QWK =', oof_qwk)
print('OOF round-QWK =', cohen_kappa_score(y_class, np.clip(np.round(oof_scores), 1, 5).astype(int), weights='quadratic'))
print('OOF MAE =', mean_absolute_error(y_class, oof_pred))
print('OOF macro-F1 =', f1_score(y_class, oof_pred, average='macro'))
print('thresholds =', thresholds.tolist())
print('combined test dist =', {int(k): int(v) for k, v in pd.Series(combined_pred).value_counts().sort_index().items()})
print('test_L1 =', test_l1)

## 9. Save artefacts

In [ ]:
metrics = {
    'method': 'pairwise_e5_cross_encoder_reranker',
    'model': MODEL_NAME,
    'folds': FOLDS,
    'seeds': SEEDS,
    'epochs': EPOCHS,
    'max_len': MAX_LEN,
    'batch_pair': BATCH_PAIR,
    'grad_accum_steps': GRAD_ACCUM_STEPS,
    'pairs_per_epoch': PAIRS_PER_EPOCH,
    'rank_loss_weight': RANK_LOSS_WEIGHT,
    'pointwise_loss_weight': POINTWISE_LOSS_WEIGHT,
    'amp_dtype': 'bf16' if USE_BF16 else 'fp16',
    'oof_qwk': float(oof_qwk),
    'oof_round_qwk': float(cohen_kappa_score(y_class, np.clip(np.round(oof_scores), 1, 5).astype(int), weights='quadratic')),
    'oof_mae': float(mean_absolute_error(y_class, oof_pred)),
    'oof_macro_f1': float(f1_score(y_class, oof_pred, average='macro')),
    'test_l1': test_l1,
    'thresholds': [float(v) for v in thresholds],
    'label_distribution_combined': {int(k): int(v) for k, v in pd.Series(combined_pred).value_counts().sort_index().items()},
    'label_distribution_public': {int(k): int(v) for k, v in pd.Series(public_pred).value_counts().sort_index().items()},
    'label_distribution_private': {int(k): int(v) for k, v in pd.Series(private_pred).value_counts().sort_index().items()},
    'fold_log': fold_log,
}
(RUN_DIR / 'metrics.json').write_text(json.dumps(metrics, indent=2))
print(json.dumps(metrics, indent=2))

pd.DataFrame({'id': train_full['id'], 'Label': y_class, 'oof_score': oof_scores, 'oof_pred': oof_pred}).to_csv(RUN_DIR / 'oof_scores.csv', index=False)
pd.DataFrame({'id': public_full['id'], 'score': public_scores, 'pred': public_pred}).to_csv(RUN_DIR / 'public_scores.csv', index=False)
pd.DataFrame({'id': private_full['id'], 'score': private_scores, 'pred': private_pred}).to_csv(RUN_DIR / 'private_scores.csv', index=False)
combo = pd.concat([pd.DataFrame({'id': public_full['id'], 'Label': public_pred}), pd.DataFrame({'id': private_full['id'], 'Label': private_pred})], ignore_index=True)
submission = sample[['id']].merge(combo, on='id', how='left')
submission['Label'] = submission['Label'].astype(int)
submission.to_csv(RUN_DIR / 'pairwise_e5_reranker_submission.csv', index=False)
print('saved to', RUN_DIR)

## 10. Zip + download

In [ ]:
zip_out = OUT / 'pairwise_e5_reranker_outputs.zip'
with zipfile.ZipFile(zip_out, 'w', zipfile.ZIP_DEFLATED) as zf:
    for p in RUN_DIR.iterdir():
        zf.write(p, arcname=f'pairwise_e5_reranker/{p.name}')
print('zipped:', zip_out, 'size MB =', round(zip_out.stat().st_size / 1e6, 2))
if PLATFORM == 'colab':
    from google.colab import files
    files.download(str(zip_out))
elif PLATFORM == 'kaggle':
    import shutil
    target = pathlib.Path('/kaggle/working') / zip_out.name
    if zip_out.resolve() != target.resolve():
        shutil.copy(zip_out, target)
    print('available at:', target)
else:
    print('local zip:', zip_out)